## 과제1.

- 완전한 에이전트 루프를 갖춘 Movie Agent를 구축하세요!
- 에이전트가 수행해야 할 작업:
  - 영화에 대한 사용자 질문을 받습니다.
  - 어떤 도구를 호출할지 결정합니다 (필요한 경우).
  - 실제로 API를 호출하여 LLM에게 실제 결과를 반환합니다.
  - LLM이 최종 답변을 줄 때까지 루프를 계속합니다.

### API 참고

- 기본 URL: https://nomad-movies.nomadcoders.workers.dev
  ![](https://i.ibb.co/1tdNgLtw/SCR-20260219-nnsv.png)

### 요구사항

- 실제 API를 호출하는 최소 3개의 도구를 갖춘 에이전트를 만드세요:
  - get_popular_movies() - /movies 호출
  - get_movie_details(id) - /movies/:id 호출
  - get_similar_movies(id) - /movies/:id/similar 호출
- 에이전트가 갖춰야 할 조건:
  - (수동 프롬프팅이 아닌) OpenAI tools 파라미터를 사용하세요.
  - 응답에서 tool_calls를 처리하세요.
  - 실제 API를 호출하고 결과를 다시 전달해야 합니다.
  - 메모리를 활용한 멀티턴 대화를 지원해야 합니다.

### 예시 상호작용

```
User: 지금 인기 있는 영화 알려줘
Agent: [get_popular_movies() 호출]
Agent: 현재 인기 영화 목록입니다: 1. 듄: 파트 2 (ID: 693134)...

User: 듄에 대해 더 알려줘
Agent: [get_movie_details(693134) 호출]
Agent: 듄: 파트 2는 드니 빌뇌브 감독의 2024년 SF 대작으로...

User: 비슷한 영화 추천해 줄래?
Agent: [get_similar_movies(693134) 호출]
Agent: 듄을 좋아하셨다면 이런 영화를 추천드립니다: 블레이드 러너 2049, 어라이벌...
```


In [13]:
import openai, json
import requests

# .env 의 OPENAI_API_KEY 를 자동으로 읽어서, openai.OpenAI() 객체 생성
client = openai.OpenAI()
messages = []

BASE_URL = "https://nomad-movies.nomadcoders.workers.dev"

In [14]:
GREEN = "\033[32m"
RED = "\033[31m"
BLUE = "\033[34m"
YELLOW = "\033[33m"
PINK = "\033[35m"
RESET = "\033[0m"

In [15]:
def get_popular_movies():
    response = requests.get(f"{BASE_URL}/movies")
    return response.json()


def get_movie_details(id):
    response = requests.get(f"{BASE_URL}/movies/{id}")
    return response.json()


def get_movie_credits(id):
    response = requests.get(f"{BASE_URL}/movies/{id}/credits")
    return response.json()


def get_similar_movies(id):
    response = requests.get(f"{BASE_URL}/movies/{id}/similar")
    return response.json()


FUNCTION_MAP = {
    "get_popular_movies": get_popular_movies,
    "get_movie_details": get_movie_details,
    "get_movie_credits": get_movie_credits,
    "get_similar_movies": get_similar_movies,
}

In [16]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "A function to get the popular movies",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": [],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "A function to get the details of a movie by its ID",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "integer",
                        "description": "The ID of the movie",
                    }
                },
                "required": ["id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_credits",
            "description": "A function to get the cast and crew of a movie by its ID",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "integer",
                        "description": "The ID of the movie",
                    }
                },
                "required": ["id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_similar_movies",
            "description": "A function to get similar movies by a movie ID",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "integer",
                        "description": "The ID of the movie",
                    }
                },
                "required": ["id"],
            },
        },
    },
]

In [17]:
from openai.types.chat import ChatCompletionMessage


def process_ai_response(message: ChatCompletionMessage):
    if message.tool_calls:
        messages.append(
            {
                "role": "assistant",
                "content": message.content or "",
                "tool_calls": [
                    {
                        "id": tool_call.id,
                        "type": "function",
                        "function": {
                            "name": tool_call.function.name,
                            "arguments": tool_call.function.arguments,
                        },
                    }
                    for tool_call in message.tool_calls
                ],
            }
        )

        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            arguments = tool_call.function.arguments

            print(GREEN + f"Agent: [{function_name} with {arguments} 호출]" + RESET)

            try:
                arguments = json.loads(arguments)
            except json.JSONDecodeError:
                arguments = {}

            function_to_run = FUNCTION_MAP.get(function_name)

            result = function_to_run(**arguments)

            # print(
            #     GREEN
            #     + f"Ran {function_name} with args {arguments} for a result of {result}"
            #     + RESET
            # )

            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": function_name,
                    "content": (
                        json.dumps(result) if not isinstance(result, str) else result
                    ),
                }
            )

        call_ai()  # 모든 tool 결과를 추가한 후에 한 번만 호출
    else:
        messages.append({"role": "assistant", "content": message.content})
        print("=" * 30)
        print(BLUE + f"Agent: {message.content}" + RESET)


def call_ai():
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=TOOLS,
    )
    process_ai_response(response.choices[0].message)

In [ ]:
while True:
    message = input("Send a message to the LLM...")
    if message == "quit" or message == "q":
        break
    # 셀 실행희 Enter 값이 stdin 버퍼에 남아 발생하는 문제 방지
    elif not message.strip():
        continue
    else:
        messages.append({"role": "user", "content": message})
        print("=" * 30)
        print(YELLOW + f"User: {message}" + RESET)
        call_ai()

User: 지금 인기 있는 영화 알려줘
Agent: [get_popular_movies with {} 호출]
Agent: 현재 인기 있는 영화 목록입니다:

1. **Shelter**
   - 개봉일: 2026년 1월 28일
   - 평점: 6.94
   - 개요: 한 남자가 섬에서 고립된 삶을 살던 중 폭풍에서 어린 소녀를 구하면서 벌어지는 일들을 다룹니다.
   - ![포스터](https://image.tmdb.org/t/p/w780/buPFnHZ3xQy6vZEHxbHgL1Pc6CR.jpg)

2. **Mercy**
   - 개봉일: 2026년 1월 20일
   - 평점: 7.125
   - 개요: 가까운 미래, 한 형사가 아내 살인 혐의로 재판을 받으면서 진실을 밝혀내야 하는 이야기입니다.
   - ![포스터](https://image.tmdb.org/t/p/w780/pyok1kZJCfyuFapYXzHcy7BLlQa.jpg)

3. **The Orphans (Les Orphelins)**
   - 개봉일: 2025년 8월 20일
   - 평점: 6.096
   - 개요: 두 친구가 고아원을 떠나 각기 다른 삶을 살던 중, 고아의 딸이 죽은 사건을 통해 다시 만나는 이야기입니다.
   - ![포스터](https://image.tmdb.org/t/p/w780/hP7mjZr2SVfjAorlRHTdV1XZmHY.jpg)

4. **The Bluff**
   - 개봉일: 2026년 2월 17일
   - 평점: 5.98
   - 개요: 한 섬에 살던 전 해적이 복수심에 불타는 옛 선장과의 대결을 통해 가족을 지키는 이야기입니다.
   - ![포스터](https://image.tmdb.org/t/p/w780/sojEzvfxR2DBcDSJyAisX8TWjov.jpg)

5. **Hellfire**
   - 개봉일: 2026년 2월 5일
   - 평점: 7.118
   - 개요: 한 방랑자가 조용한 마을에 도착하여 범죄 보스에 맞서 싸우는 이야기입니다.
   - ![포스